In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BUDGET = 0.05
GARBAGE_RATIO = 0.05
K_NN = 15

print(f" Environment ready. Using: {DEVICE}")

# Load DINOv2 (The 'Zero-Shot' Vision Encoder)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(DEVICE)
model.eval()
print(" DINOv2 loaded successfully.")

 Environment ready. Using: cuda
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 283MB/s]


 DINOv2 loaded successfully.


In [3]:

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

def get_features(dataset, title):

    loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=False)
    feats, lbls = [], []
    with torch.no_grad():
        for imgs, l in tqdm(loader, desc=title):

            emb = model(imgs.to(DEVICE)).cpu().numpy()
            feats.append(emb)
            lbls.append(l.numpy())
    return np.concatenate(feats), np.concatenate(lbls)

print("Starting feature extraction process")

X_train_clean, y_train = get_features(train_ds, "Processing Training Set")
X_test, y_test = get_features(test_ds, "Processing Test Set")

X_train_dirty = X_train_clean.copy()
n_garbage = int(len(X_train_clean) * GARBAGE_RATIO)
garbage_idx = np.random.choice(len(X_train_clean), n_garbage, replace=False)

X_train_dirty[garbage_idx] = np.random.normal(loc=15.0, scale=3.0, size=(n_garbage, X_train_clean.shape[1]))

print(f"- Clean Features Shape: {X_train_clean.shape}")
print(f"- Dirty Features created with {GARBAGE_RATIO*100}% injected noise.")

Starting feature extraction process


Processing Test Set: 100%|██████████| 79/79 [01:09<00:00,  1.13it/s]

- Clean Features Shape: (50000, 384)
- Dirty Features created with 5.0% injected noise.


In [4]:
def iterative_subspace_sampling(X, n_select, n_subspaces=10):

    n_samples, d = X.shape
    subspace_dim = d // n_subspaces

    indices = [np.random.randint(n_samples)]

    min_dists = np.full(n_samples, np.inf)

    for i in tqdm(range(n_select - 1), desc="ZCore Sampling", leave=False):

        start = (i % n_subspaces) * subspace_dim
        end = start + subspace_dim

        last_added_vec = X[indices[-1], start:end]
        current_subspace_data = X[:, start:end]

        dists = np.linalg.norm(current_subspace_data - last_added_vec, axis=1)

        min_dists = np.minimum(min_dists, dists)

        next_idx = np.argmax(min_dists)
        indices.append(next_idx)

    return np.array(indices)

def our_robust_zcore(X, n_select, k=15):

    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    dists, _ = nbrs.kneighbors(X)
    avg_dist = np.mean(dists, axis=1)

    threshold = np.percentile(avg_dist, 100 * (1 - GARBAGE_RATIO))
    inlier_mask = avg_dist < threshold

    X_clean = X[inlier_mask]
    clean_indices = np.where(inlier_mask)[0]

    selected_clean_idx = iterative_subspace_sampling(X_clean, n_select)

    return clean_indices[selected_clean_idx]

print(" Sampling algorithms defined.")

 Sampling algorithms defined.


In [5]:
n_to_pick = int(len(X_train_dirty) * BUDGET)

clf_upper = LogisticRegression(max_iter=500).fit(X_train_clean, y_train)
acc_upper = accuracy_score(y_test, clf_upper.predict(X_test))

idx_rand = np.random.choice(len(X_train_dirty), n_to_pick, replace=False)
clf_rand = LogisticRegression(max_iter=500).fit(X_train_dirty[idx_rand], y_train[idx_rand])
acc_rand = accuracy_score(y_test, clf_rand.predict(X_test))

idx_paper = iterative_subspace_sampling(X_train_dirty, n_to_pick)
clf_paper = LogisticRegression(max_iter=500).fit(X_train_dirty[idx_paper], y_train[idx_paper])
acc_paper = accuracy_score(y_test, clf_paper.predict(X_test))

idx_robust = our_robust_zcore(X_train_dirty, n_to_pick, K_NN)
clf_robust = LogisticRegression(max_iter=500).fit(X_train_dirty[idx_robust], y_train[idx_robust])
acc_robust = accuracy_score(y_test, clf_robust.predict(X_test))

print("\n" + "="*60)
print(f"{'Methodology Scenario':<40} | {'Accuracy (%)':<12}")
print("-" * 60)
print(f"{'Upper Bound (Full Clean Data)':<40} | {acc_upper*100:<12.2f}")
print(f"{'Random Selection (5% Coreset)':<40} | {acc_rand*100:<12.2f}")
print(f"{'Official ZCore (Paper Method)':<40} | {acc_paper*100:<12.2f}")
print(f"{'Robust ZCore (Our Extension)':<40} | {acc_robust*100:<12.2f}")
print("-" * 60)
print(f"{'Relative Performance (Ours vs Full)':<40} | {(acc_robust/acc_upper)*100:<12.2f}%")
print(f"{'Absolute Gain (Ours vs Paper)':<40} | {(acc_robust-acc_paper)*100:<12.2f}%")
print("="*60)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Methodology Scenario                     | Accuracy (%)
------------------------------------------------------------
Upper Bound (Full Clean Data)            | 94.80       
Random Selection (5% Coreset)            | 93.15       
Official ZCore (Paper Method)            | 49.35       
Robust ZCore (Our Extension)             | 93.44       
------------------------------------------------------------
Relative Performance (Ours vs Full)      | 98.57       %
Absolute Gain (Ours vs Paper)            | 44.09       %
